# Tabla 5.1 — Comparación de Datasets de Edificaciones
**Responsable:** Andrés  
**Capítulo:** 5.4 del informe `informe_upme_solar.tex`  
**Salida:** `semana_3/outputs/tables/comparacion_datasets.csv`

Este notebook consulta las colecciones `ms_buildings` y `goo_buildings` en MongoDB
y genera automáticamente la tabla comparativa de la Sección 5.4.  
Si alguna colección no existe o está vacía imprime un `WARNING` claro.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import warnings
from datetime import datetime

# ── Raíz del proyecto ──────────────────────────────────────────
ROOT = Path("__file__").resolve().parent.parent.parent
sys.path.append(str(ROOT))
from config import get_db, BASE

# ── Carpeta de salida ──────────────────────────────────────────
OUTPUT_DIR = Path(BASE) / "semana_3" / "outputs" / "tables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / "comparacion_datasets.csv"

print(f"Conectando a MongoDB...")
db = get_db()
print(f"Base de datos: {db.name}")
print(f"Colecciones existentes: {db.list_collection_names()}")

In [ ]:
# ── Helpers ────────────────────────────────────────────────────

def coleccion_ok(nombre: str) -> bool:
    """Devuelve True si la colección existe y tiene al menos 1 documento."""
    if nombre not in db.list_collection_names():
        print(f"\n⚠️  WARNING: PENDIENTE — la colección '{nombre}' NO existe en MongoDB.")
        print(f"    Verifica que el responsable haya ejecutado el script de carga.")
        return False
    n = db[nombre].count_documents({})
    if n == 0:
        print(f"\n⚠️  WARNING: PENDIENTE — la colección '{nombre}' está VACÍA.")
        print(f"    Verifica que el responsable haya ejecutado el script de carga.")
        return False
    return True


def get_stats(nombre: str) -> dict:
    """Retorna estadísticas clave de una colección."""
    if not coleccion_ok(nombre):
        responsable = "Juan Pablo" if nombre == "ms_buildings" else "Jineth"
        return {
            "total_edificaciones": f"PENDIENTE: {responsable}",
            "tamaño_en_disco_MB":  f"PENDIENTE: {responsable}",
            "tiene_atributo_area": f"PENDIENTE: {responsable}",
            "tiempo_de_carga":     f"PENDIENTE: {responsable}",
        }

    col = db[nombre]

    # Total de documentos
    total = col.count_documents({})

    # Tamaño en disco via collStats
    stats_cmd = db.command("collStats", nombre)
    size_mb = round(stats_cmd.get("storageSize", 0) / (1024 ** 2), 2)

    # Verificar si existe campo area_m2
    sample = col.find_one({"area_m2": {"$exists": True}})
    tiene_area = "Sí (area_m2)" if sample else "No"

    return {
        "total_edificaciones": total,
        "tamaño_en_disco_MB":  size_mb,
        "tiene_atributo_area": tiene_area,
        "tiempo_de_carga":     "ver log de carga",   # no se puede recuperar a posteriori
    }

print("Helpers definidos.")

In [ ]:
# ── Consultar colecciones ──────────────────────────────────────
print("=" * 55)
print("Consultando ms_buildings...")
ms_stats = get_stats("ms_buildings")

print("\nConsultando goo_buildings...")
goo_stats = get_stats("goo_buildings")

print("\nEstadísticas crudas:")
print("  ms_buildings :", ms_stats)
print("  goo_buildings:", goo_stats)

In [ ]:
# ── Construir tabla comparativa ────────────────────────────────
#
# Campos fijos (documentados / conocidos de antemano)
AÑO_MS  = "2014–2021"
AÑO_GOO = "PENDIENTE: Jineth"   # completar cuando Jineth confirme
LIC_MS  = "ODbL"
LIC_GOO = "CC BY-4.0 / ODbL"

tabla = pd.DataFrame([
    {
        "aspecto":              "total_edificaciones",
        "microsoft":            ms_stats["total_edificaciones"],
        "google":               goo_stats["total_edificaciones"],
    },
    {
        "aspecto":              "tamaño_en_disco_MB",
        "microsoft":            ms_stats["tamaño_en_disco_MB"],
        "google":               goo_stats["tamaño_en_disco_MB"],
    },
    {
        "aspecto":              "año_imágenes",
        "microsoft":            AÑO_MS,
        "google":               AÑO_GOO,
    },
    {
        "aspecto":              "tiene_atributo_área",
        "microsoft":            ms_stats["tiene_atributo_area"],
        "google":               goo_stats["tiene_atributo_area"],
    },
    {
        "aspecto":              "licencia",
        "microsoft":            LIC_MS,
        "google":               LIC_GOO,
    },
    {
        "aspecto":              "tiempo_de_carga",
        "microsoft":            ms_stats["tiempo_de_carga"],
        "google":               goo_stats["tiempo_de_carga"],
    },
])

tabla["generado_en"] = datetime.utcnow().strftime("%Y-%m-%d %H:%M UTC")

print("\nTabla 5.1 — Comparación de datasets")
print(tabla.to_string(index=False))

In [ ]:
# ── Exportar a CSV ─────────────────────────────────────────────
tabla.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")
print(f"\n✅ Tabla exportada → {OUTPUT_PATH}")

# ── Mostrar resumen legible ────────────────────────────────────
print("\n" + "=" * 55)
print("RESUMEN PARA EL INFORME (Tabla 5.1)")
print("=" * 55)
for _, row in tabla.iterrows():
    print(f"  {row['aspecto']:<25} | MS: {str(row['microsoft']):<30} | GOO: {row['google']}")

# ── Advertencia si hay pendientes ─────────────────────────────
pendientes = tabla[
    tabla.apply(lambda r: "PENDIENTE" in str(r["microsoft"]) or "PENDIENTE" in str(r["google"]), axis=1)
]
if not pendientes.empty:
    print("\n" + "⚠️  " * 10)
    print("CELDAS PENDIENTES EN LA TABLA:")
    for _, row in pendientes.iterrows():
        print(f"  → {row['aspecto']}: MS={row['microsoft']} | GOO={row['google']}")
    print("    Completar antes de entregar el informe.")
else:
    print("\n✅ No hay celdas pendientes — tabla completa.")